# RadiScan — training on Colab / Kaggle GPU

A free T4 gets you to a defensible mean AUC in a couple of hours; a laptop CPU does not.
The pattern this notebook follows is the one to use: **train in the cloud, run the app locally.**

Runtime → Change runtime type → **T4 GPU** before you start.


## 1. Setup

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Clone your fork and install. Torch is already present on Colab, so --no-deps
# on the package install keeps it from being replaced.
!git clone https://github.com/<your-username>/radiscan.git
%cd radiscan
!pip install -q timm albumentations opencv-python-headless pydicom pydantic-settings
!pip install -q --no-deps -e .

import radiscan
print("radiscan", radiscan.__version__)

## 2. Data

CheXpert-small is ~11 GB. Put your `kaggle.json` in place first
(Kaggle → Account → Create New API Token).

In [ ]:
from google.colab import files
files.upload()  # kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d ashery/chexpert-small -p data/raw/chexpert
!cd data/raw/chexpert && unzip -q chexpert-small.zip && rm chexpert-small.zip
!ls data/raw/chexpert

### Patient-disjoint splits

CheXpert has several studies per patient. A random row split leaks patients
between train and validation and inflates AUC by 3–6 points — the single most
common silent mistake in student chest X-ray projects.

In [ ]:
!python -m radiscan.cli.prepare_data \
  --source data/raw/chexpert/CheXpert-v1.0-small/train.csv \
  --out-dir data/splits --frontal-only \
  --train-size 40000 --val-size 4000 --test-size 4000

### Verify before training

Three seconds. It confirms CSV paths resolve to files on disk and prints
per-class positive rates. Never skip it.

In [ ]:
!python -m radiscan.cli.verify_data --config configs/chexpert_base.yaml

## 3. Train

Batch 32 fits a 16 GB T4 at 320px with AMP on. If you hit OOM, halve
`train.batch_size` and set `train.accumulate_steps=2` so the *effective* batch
stays 32 — the schedule is tuned for that.

In [ ]:
!python -m radiscan.cli.train --config configs/chexpert_base.yaml \
  --set data.num_workers=2 train.batch_size=32 train.epochs=25

Colab disconnects. Checkpoint after every epoch to Drive so a dropped
session costs you one epoch, not the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/radiscan
!cp models/checkpoints/best_model.pth /content/drive/MyDrive/radiscan/
!cp -r logs /content/drive/MyDrive/radiscan/

## 4. Evaluate on the held-out test split

One command produces every table and figure the report needs, each PNG paired
with the CSV behind it.

In [ ]:
!python -m radiscan.cli.evaluate --config configs/chexpert_base.yaml --split test --out results

In [ ]:
import pandas as pd
df = pd.read_csv('results/per_class_metrics.csv')
df[['label','support','auc','ap','precision','recall','f1']].round(4)

In [ ]:
from IPython.display import Image, display
for name in ['per_class_auc','roc_curves','calibration','uncertainty_quartiles','selective_prediction','training_curves']:
    print(name)
    display(Image(f'results/{name}.png'))

### The table that justifies the uncertainty component

If accuracy falls monotonically from Q1 to Q4, the MC-Dropout estimates carry
real information and the whole uncertainty half of the project is validated in
one table. If it does not, report that honestly — an examiner respects a clean
negative result far more than a fudged positive one.

In [ ]:
pd.read_csv('results/uncertainty_quartiles.csv')

## 5. Ablation (week 5)

Change **only** the backbone. Same seed, same data, same schedule — otherwise
Table 2 is not a controlled experiment.

In [ ]:
!python -m radiscan.cli.train --config configs/ablation_convnext.yaml --set data.num_workers=2
!python -m radiscan.cli.evaluate --config configs/ablation_convnext.yaml --split test --out results/convnext

!python -m radiscan.cli.train --config configs/chexpert_base.yaml \
  --set model.use_transformer=true train.batch_size=16 train.checkpoint_dir=models/checkpoints/hybrid
!python -m radiscan.cli.evaluate --config configs/chexpert_base.yaml --split test \
  --checkpoint models/checkpoints/hybrid/best_model.pth --out results/hybrid

In [ ]:
import json, pandas as pd

rows = []
for name, path in [('DenseNet121','results'), ('ConvNeXt-Tiny','results/convnext'), ('DenseNet+ViT','results/hybrid')]:
    try:
        s = json.load(open(f'{path}/summary.json'))
        rows.append({'configuration': name,
                     'mean AUC': round(s['overall']['mean_auc'], 4),
                     'mean AP': round(s['overall']['mean_ap'], 4),
                     'CheXpert-5 AUC': round(s['chexpert_competition_5']['mean_auc'], 4),
                     'ECE': round(s['ece'], 4)})
    except FileNotFoundError:
        pass
rows.append({'configuration': 'CheXNet (published, NIH-14)', 'mean AUC': 0.841,
             'mean AP': None, 'CheXpert-5 AUC': None, 'ECE': None})
pd.DataFrame(rows)

## 6. Take the checkpoint home

Download `best_model.pth` into `models/checkpoints/` on your laptop. Inference
runs fine on CPU (roughly 2–5 s per image with 20 MC passes), so the whole
application is demonstrable without a GPU:

```bash
uvicorn services.api.main:app --reload
cd web && npm run dev
```


In [ ]:
from google.colab import files
files.download('models/checkpoints/best_model.pth')